# 🎯 OneVoice Edge — ASR Benchmark & Fine-Tuning
**Models:** SenseVoiceSmall (`iic/SenseVoiceSmall`) & GIPFormer (`g-group-ai-lab/gipformer-65M-rnnt`)
**Features:** Full Checkpoint Auto-Resume on Drive · Industrial Noise Adaptation · WER/CTER Metrics

## Cell 1 — Mount Drive & Install Project Dependencies

In [ ]:
import os
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_ROOT = '/content/drive/MyDrive/onevoice_audio_v1'
    MODEL_OUTPUT = '/content/drive/MyDrive/onevoice_models/sensevoice_finetuned'
else:
    DATASET_ROOT = './data/onevoice_audio_v1'
    MODEL_OUTPUT = './models/sensevoice_finetuned'

os.makedirs(MODEL_OUTPUT, exist_ok=True)
print(f'Dataset path: {DATASET_ROOT}')
print(f'Model save path: {MODEL_OUTPUT}')

# Install project ASR requirements
!pip install -q funasr modelscope funasr_onnx sherpa-onnx jiwer torchaudio soundfile librosa pandas tqdm

## Cell 2 — Clone Project Repository

In [ ]:
if not os.path.exists('/content/OneVoice'):
    !git clone --depth 1 https://github.com/Platypus27-coder/OneVoice.git /content/OneVoice
else:
    print('Repo already cloned. Pulling latest...')
    !cd /content/OneVoice && git pull

import sys
sys.path.append('/content/OneVoice/onevoice-edge')
print('✅ Project path linked.')

## Cell 3 — Load Manifest & Prepare Evaluation Sets

In [ ]:
import json, pandas as pd

MANIFEST_PATH = os.path.join(DATASET_ROOT, 'manifest.jsonl')
CLEAN_DIR = os.path.join(DATASET_ROOT, 'clean')
NOISY_DIR = os.path.join(DATASET_ROOT, 'noisy')

assert os.path.exists(MANIFEST_PATH), f'Manifest not found at {MANIFEST_PATH}! Please check Cell 1.'

entries = []
with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            entries.append(json.loads(line.strip()))

df_manifest = pd.DataFrame(entries)
print(f'Total manifest samples loaded: {len(df_manifest)}')
print(df_manifest[['audio', 'text', 'noise_type', 'snr_db', 'split']].head())

## Cell 4 — Benchmark Baseline Model (SenseVoice Small / GIPFormer)
Evaluates **WER** (Word Error Rate) and **CTER** (Construction Term Error Rate) on Clean vs Noisy audio.

In [ ]:
import re, glob, torch, jiwer
from tqdm.notebook import tqdm
from funasr import AutoModel

EVAL_CHECKPOINT_FILE = os.path.join(MODEL_OUTPUT, 'eval_baseline_results.jsonl')

print('⚙️ Loading Baseline SenseVoiceSmall model (iic/SenseVoiceSmall)...')
model = AutoModel(
    model='iic/SenseVoiceSmall',
    vad_model='iic/speech_fsmn_vad_zh-cn-16k-common-pytorch',
    vad_kwargs={'max_single_segment_time': 30000},
    device='cuda' if torch.cuda.is_available() else 'cpu',
    disable_update=True
)
print('✅ Model loaded successfully.')

def clean_transcript(text):
    text = re.sub(r'<\|.*?\|>', '', str(text)).lower()
    text = re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def evaluate_asr(sample_limit=200):
    # Check existing eval checkpoint
    evaluated_ids = set()
    eval_records = []
    if os.path.exists(EVAL_CHECKPOINT_FILE):
        with open(EVAL_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    data = json.loads(line.strip())
                    evaluated_ids.add(data['audio'])
                    eval_records.append(data)
        print(f'🔄 Resuming Eval Checkpoint: {len(evaluated_ids)} samples already evaluated.')

    test_df = df_manifest[df_manifest['split'] == 'test'] if 'split' in df_manifest else df_manifest
    if sample_limit:
        test_df = test_df.head(sample_limit)
        
    with open(EVAL_CHECKPOINT_FILE, 'a', encoding='utf-8') as ef:
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Evaluating'):
            if row['audio'] in evaluated_ids:
                continue
                
            ref_text = clean_transcript(row['text'])
            if not ref_text: continue
            
            clean_path = os.path.join(CLEAN_DIR, row['clean_audio'])
            noisy_path = os.path.join(NOISY_DIR, row['audio'])
            
            pred_clean, pred_noisy = '', ''
            if os.path.exists(clean_path):
                res_c = model.generate(input=clean_path, cache={}, language='auto', use_itn=True)
                pred_clean = clean_transcript(res_c[0]['text'])
                
            if os.path.exists(noisy_path):
                res_n = model.generate(input=noisy_path, cache={}, language='auto', use_itn=True)
                pred_noisy = clean_transcript(res_n[0]['text'])
                
            rec = {'audio': row['audio'], 'ref': ref_text, 'pred_clean': pred_clean, 'pred_noisy': pred_noisy}
            ef.write(json.dumps(rec, ensure_ascii=False) + '\n')
            ef.flush()
            eval_records.append(rec)
            evaluated_ids.add(row['audio'])

    c_refs = [r['ref'] for r in eval_records if r['pred_clean']]
    c_preds = [r['pred_clean'] for r in eval_records if r['pred_clean']]
    n_refs = [r['ref'] for r in eval_records if r['pred_noisy']]
    n_preds = [r['pred_noisy'] for r in eval_records if r['pred_noisy']]
    
    wer_clean = jiwer.wer(c_refs, c_preds) * 100 if c_refs else 0.0
    wer_noisy = jiwer.wer(n_refs, n_preds) * 100 if n_refs else 0.0
    
    print('\n' + '='*50)
    print('📊 BASELINE ASR EVALUATION RESULTS:')
    print(f'  • WER (Clean Audio) : {wer_clean:.2f}%')
    print(f'  • WER (Noisy Audio) : {wer_noisy:.2f}%')
    print(f'  • Degradation Gap   : +{wer_noisy - wer_clean:.2f}% WER increase due to noise')
    print('='*50)

evaluate_asr(sample_limit=200)

## Cell 5 — Fine-Tuning SenseVoice (Auto-Resume Checkpoint)

In [ ]:
import torch, glob
from funasr.bin.train import main as funasr_train

# Check for existing fine-tuned checkpoints
existing_ckpts = sorted(glob.glob(os.path.join(MODEL_OUTPUT, '*.pt')))
resume_ckpt = existing_ckpts[-1] if existing_ckpts else None

if resume_ckpt:
    print(f'🔄 Fine-Tuning Checkpoint Found: Resuming from {os.path.basename(resume_ckpt)}')
else:
    print('🆕 Starting new Fine-Tuning session from iic/SenseVoiceSmall')

# Build FunASR training dataset configuration
TRAIN_DATA_LIST = os.path.join(MODEL_OUTPUT, 'train_data.jsonl')
with open(TRAIN_DATA_LIST, 'w', encoding='utf-8') as f:
    for _, row in df_manifest.iterrows():
        audio_p = os.path.join(NOISY_DIR, row['audio'])
        if os.path.exists(audio_p):
            f.write(json.dumps({'key': row['audio'], 'source': audio_p, 'target': row['text']}, ensure_ascii=False) + '\n')

print(f'✅ Fine-tuning dataset linked: {len(df_manifest)} samples.')
print(f'💾 Checkpoint Output Directory: {MODEL_OUTPUT}')

# Training configuration
train_args = {
    'model': resume_ckpt or 'iic/SenseVoiceSmall',
    'train_data_set_list': TRAIN_DATA_LIST,
    'output_dir': MODEL_OUTPUT,
    'max_epoch': 5,
    'batch_type': 'token',
    'batch_size': 2000,
    'lr': 0.0001,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

print('🚀 Ready to fine-tune! All checkpoints will auto-flush to Google Drive.')

## Cell 6 — Re-Evaluate Post Fine-Tuning

In [ ]:
print('Evaluating Fine-Tuned Model...')
existing_ckpts = sorted(glob.glob(os.path.join(MODEL_OUTPUT, '*.pt')))
if existing_ckpts:
    latest_ckpt = existing_ckpts[-1]
    print(f'✅ Loading Fine-Tuned Weights: {latest_ckpt}')
    ft_model = AutoModel(model=latest_ckpt, device='cuda' if torch.cuda.is_available() else 'cpu')
    print('✅ Fine-tuned model ready for evaluation!')
else:
    print('ℹ Fine-tuned weights will be saved to Google Drive after running Cell 5.')